# 2. Tối ưu hoá dựa trên chi phí (Cost-Based Optimization – CBO)

Theo sách, Chương 19, mục 19.3 đến 19.6.

**Ý tưởng:** khác với tối ưu theo quy tắc kinh nghiệm (notebook 1) chỉ áp dụng các luật biến đổi cố định, bộ tối ưu dựa trên chi phí
**liệt kê nhiều kế hoạch thực thi**, dùng **thống kê trong catalog** để **ước lượng chi phí** (số khối đĩa phải đọc/ghi) của từng kế hoạch,
rồi chọn kế hoạch rẻ nhất.

**Mục tiêu**

| Phần | Nội dung | Mục trong sách |
|---|---|---|
| 1 | Danh mục thống kê (catalog): `r`, `b`, `bfr`, `NDV`, số tầng chỉ mục `x` | 19.3 |
| 2 | Ước lượng độ chọn lọc (selectivity) và số dòng kết quả | 19.3 |
| 3 | Hàm chi phí cho phép chọn S1 – S6, chọn đường truy xuất rẻ nhất | 19.4 |
| 4 | Hàm chi phí cho phép nối J1 – J4, chọn giải thuật nối và bảng ngoài | 19.5 |
| 5 | Chọn thứ tự nối bằng quy hoạch động (cây lệch trái) | 19.5, 19.6 |
| 6 | Đối chiếu với bộ tối ưu của PostgreSQL (file `2.2_cbo_postgres_demo.sql`) | |

Chi phí được tính theo **số khối đĩa truy cập** như trong sách (bỏ qua chi phí CPU).
Notebook chỉ dùng Python chuẩn, không cần cài thêm thư viện.

## 1. Danh mục thống kê (catalog)

Bộ tối ưu không đọc dữ liệu thật khi lập kế hoạch, nó chỉ dựa vào các con số lưu trong catalog:

| Ký hiệu | Ý nghĩa |
|---|---|
| `r` | số dòng của bảng |
| `b` | số khối (block) đĩa của bảng |
| `bfr` | hệ số khối: số dòng trên một khối |
| `NDV(A)` | số giá trị khác nhau của cột A |
| `x` | số tầng của chỉ mục |
| `bI1` | số khối lá (tầng 1) của chỉ mục phụ |

`EMPLOYEE` và `DEPARTMENT` lấy số liệu theo ví dụ của sách (mục 19.4 – 19.5).
`WORKS_ON` và `PROJECT` là số liệu **giả định** cùng quy mô, dùng cho phần 5.

In [1]:
import math
from itertools import combinations, permutations

# kind: primary (chỉ mục chính trên khoá thứ tự), clustering (chỉ mục cụm),
#       secondary (chỉ mục phụ B+), key = True nếu cột là khoá
CATALOG = {
    "EMPLOYEE": {"r": 10_000, "b": 2_000, "bfr": 5, "ordered_on": "Salary",
                 "ndv": {"Ssn": 10_000, "Dno": 125, "Salary": 500, "Sex": 2},
                 "index": {"Salary": {"kind": "clustering", "x": 3},
                           "Ssn":    {"kind": "secondary",  "x": 4, "key": True},
                           "Dno":    {"kind": "secondary",  "x": 2, "bI1": 4},
                           "Sex":    {"kind": "secondary",  "x": 1}}},
    "DEPARTMENT": {"r": 125, "b": 13, "bfr": 10, "ordered_on": "Dnumber",
                   "ndv": {"Dnumber": 125, "Mgr_ssn": 125, "Dname": 125},
                   "index": {"Dnumber": {"kind": "primary",   "x": 1, "key": True},
                             "Mgr_ssn": {"kind": "secondary", "x": 2, "key": True}}},
    # ---- Giả định, dùng cho phần 5 ----
    "WORKS_ON": {"r": 40_000, "b": 1_000, "bfr": 40, "ordered_on": "Essn",
                 "ndv": {"Essn": 10_000, "Pno": 2_000},
                 "index": {"Essn": {"kind": "clustering", "x": 3}}},   # khoá chính (Essn, Pno)
    "PROJECT": {"r": 2_000, "b": 200, "bfr": 10, "ordered_on": "Pnumber",
                "ndv": {"Pnumber": 2_000, "Pname": 2_000, "Plocation": 200, "Dnum": 125},
                "index": {"Pnumber":   {"kind": "primary",   "x": 2, "key": True},
                          "Plocation": {"kind": "secondary", "x": 2, "bI1": 10}}},
}

print(f"{'Bảng':<12}{'r':>8}{'b':>7}{'bfr':>5}   Chỉ mục")
for name, t in CATALOG.items():
    idx = ", ".join(f"{c} ({i['kind']}, x={i['x']})" for c, i in t["index"].items())
    print(f"{name:<12}{t['r']:>8,}{t['b']:>7,}{t['bfr']:>5}   {idx}")

Bảng               r      b  bfr   Chỉ mục
EMPLOYEE      10,000  2,000    5   Salary (clustering, x=3), Ssn (secondary, x=4), Dno (secondary, x=2), Sex (secondary, x=1)
DEPARTMENT       125     13   10   Dnumber (primary, x=1), Mgr_ssn (secondary, x=2)
WORKS_ON      40,000  1,000   40   Essn (clustering, x=3)
PROJECT        2,000    200   10   Pnumber (primary, x=2), Plocation (secondary, x=2)


## 2. Ước lượng độ chọn lọc (selectivity)

Độ chọn lọc `sl` của một điều kiện là tỉ lệ số dòng thoả điều kiện, `0 ≤ sl ≤ 1`.
Số dòng kết quả ước lượng (selection cardinality) là `s = sl × r`.

| Điều kiện | Công thức ước lượng `sl` |
|---|---|
| `A = v`, A là khoá | `1 / r` |
| `A = v`, A không là khoá | `1 / NDV(A)` (giả định phân bố đều) |
| `A > v`, `A < v` (không có histogram) | `1/2` (sách giả định khoảng một nửa số dòng) |
| `C1 AND C2` | `sl(C1) × sl(C2)` (giả định độc lập) |
| `C1 OR C2` | `sl(C1) + sl(C2) − sl(C1) × sl(C2)` |

In [2]:
def selectivity(rel, cond):
    col, op, _ = cond
    if op == "=":
        return 1 / CATALOG[rel]["ndv"][col]          # cột khoá có NDV = r nên ra 1/r
    return 0.5                                       # so sánh khoảng, không có histogram


def selectivity_and(rel, conds):
    sl = 1.0
    for c in conds:
        sl *= selectivity(rel, c)
    return sl


def show(c):
    return f"{c[0]} {c[1]} {c[2]!r}"


for c in [("Ssn", "=", "123456789"), ("Dno", "=", 5), ("Sex", "=", "F"), ("Dno", ">", 5)]:
    sl = selectivity("EMPLOYEE", c)
    print(f"{show(c):<22} sl = {sl:<8.4g} s = sl × r = {sl * CATALOG['EMPLOYEE']['r']:>7,.0f} dòng")

conds = [("Dno", "=", 5), ("Salary", ">", 30000), ("Sex", "=", "F")]
sl = selectivity_and("EMPLOYEE", conds)
print(f"\nDno = 5 AND Salary > 30000 AND Sex = 'F': sl = 1/125 × 1/2 × 1/2 = {sl:.4f}"
      f" -> s = {sl * CATALOG['EMPLOYEE']['r']:.0f} dòng")

Ssn = '123456789'      sl = 0.0001   s = sl × r =       1 dòng
Dno = 5                sl = 0.008    s = sl × r =      80 dòng
Sex = 'F'              sl = 0.5      s = sl × r =   5,000 dòng
Dno > 5                sl = 0.5      s = sl × r =   5,000 dòng

Dno = 5 AND Salary > 30000 AND Sex = 'F': sl = 1/125 × 1/2 × 1/2 = 0.0020 -> s = 20 dòng


## 3. Hàm chi phí cho phép chọn (mục 19.4)

| Ký hiệu | Phương pháp | Chi phí (số khối) |
|---|---|---|
| S1 | Quét tuyến tính | `b`; nếu điều kiện bằng trên khoá thì trung bình `b/2` |
| S2 | Tìm nhị phân (tệp sắp theo cột A) | `log2(b) + ⌈s/bfr⌉ − 1` |
| S3a | Chỉ mục chính, điều kiện bằng | `x + 1` |
| S4 | Chỉ mục có thứ tự, điều kiện khoảng | `x + b/2` |
| S5 | Chỉ mục cụm, điều kiện bằng | `x + ⌈s/bfr⌉` |
| S6 | Chỉ mục phụ B+ | bằng trên khoá: `x + 1`; bằng không khoá: `x + 1 + s`; khoảng: `x + bI1/2 + r/2` |

Với điều kiện hội (AND) nhiều vế, bộ tối ưu thử dùng đường truy xuất của **từng vế**, các vế còn lại kiểm tra trên dòng đã lấy lên trong bộ nhớ,
và chọn phương án rẻ nhất.

In [3]:
def access_paths(rel, cond):
    # Các đường truy xuất dùng được cho MỘT điều kiện, kèm chi phí
    t = CATALOG[rel]
    col, op, _ = cond
    s = math.ceil(selectivity(rel, cond) * t["r"])
    is_key = t["ndv"].get(col) == t["r"]
    paths = {}
    if t.get("ordered_on") == col and op == "=":
        paths["S2 Tìm nhị phân"] = math.log2(t["b"]) + math.ceil(s / t["bfr"]) - 1
    idx = t["index"].get(col)
    if idx:
        x = idx["x"]
        if idx["kind"] == "primary":
            paths["S3a Chỉ mục chính" if op == "=" else "S4 Chỉ mục chính (khoảng)"] = \
                x + 1 if op == "=" else x + t["b"] / 2
        elif idx["kind"] == "clustering":
            paths["S5 Chỉ mục cụm" if op == "=" else "S4 Chỉ mục cụm (khoảng)"] = \
                x + math.ceil(s / t["bfr"]) if op == "=" else x + t["b"] / 2
        elif op == "=":
            paths["S6 Chỉ mục phụ"] = x + 1 if idx.get("key") else x + 1 + s
        else:
            paths["S6 Chỉ mục phụ (khoảng)"] = x + idx.get("bI1", 0) / 2 + t["r"] / 2
    return paths


def best_select(rel, conds, verbose=True):
    t = CATALOG[rel]
    single_key_eq = (len(conds) == 1 and conds[0][1] == "="
                     and t["ndv"].get(conds[0][0]) == t["r"])
    options = [("S1 Quét tuyến tính", "", t["b"] / 2 if single_key_eq else t["b"])]
    for c in conds:
        for name, cost in access_paths(rel, c).items():
            options.append((name, f"trên {c[0]}", cost))
    best = min(options, key=lambda o: o[2])
    if verbose:
        rows = selectivity_and(rel, conds) * t["r"]
        print(f"σ[{' AND '.join(map(show, conds))}]({rel})   ước lượng {rows:,.0f} dòng")
        for name, on, cost in options:
            mark = "  <== chọn" if (name, on, cost) == best else ""
            print(f"   {name + ' ' + on:<38}{cost:>9,.1f} khối{mark}")
        print()
    return best


OPS = {
    "OP1": [("Ssn", "=", "123456789")],
    "OP2": [("Dno", ">", 5)],
    "OP3": [("Dno", "=", 5)],
    "OP4": [("Dno", "=", 5), ("Salary", ">", 30000), ("Sex", "=", "F")],
}
for op, conds in OPS.items():
    print(op, end=": ")
    best_select("EMPLOYEE", conds)

OP1: σ[Ssn = '123456789'](EMPLOYEE)   ước lượng 1 dòng
   S1 Quét tuyến tính                      1,000.0 khối
   S6 Chỉ mục phụ trên Ssn                     5.0 khối  <== chọn

OP2: σ[Dno > 5](EMPLOYEE)   ước lượng 5,000 dòng
   S1 Quét tuyến tính                      2,000.0 khối  <== chọn
   S6 Chỉ mục phụ (khoảng) trên Dno        5,004.0 khối

OP3: σ[Dno = 5](EMPLOYEE)   ước lượng 80 dòng
   S1 Quét tuyến tính                      2,000.0 khối
   S6 Chỉ mục phụ trên Dno                    83.0 khối  <== chọn

OP4: σ[Dno = 5 AND Salary > 30000 AND Sex = 'F'](EMPLOYEE)   ước lượng 20 dòng
   S1 Quét tuyến tính                      2,000.0 khối
   S6 Chỉ mục phụ trên Dno                    83.0 khối  <== chọn
   S4 Chỉ mục cụm (khoảng) trên Salary     1,003.0 khối
   S6 Chỉ mục phụ trên Sex                 5,002.0 khối



**Nhận xét:**

- **OP1**: điều kiện bằng trên khoá `Ssn`, chỉ mục phụ chỉ tốn `x + 1 = 5` khối thay vì đọc trung bình 1.000 khối.
- **OP2**: điều kiện khoảng `Dno > 5` lấy lên khoảng một nửa bảng. Qua chỉ mục phụ, mỗi dòng có thể nằm ở một khối khác nhau nên tốn hơn 5.000 khối,
  **đắt hơn quét tuyến tính** (2.000 khối). Bộ tối ưu chọn quét bảng: có chỉ mục **không có nghĩa là** luôn dùng chỉ mục.
- **OP3**: `Dno = 5` chỉ khoảng 80 dòng, chỉ mục phụ rẻ hơn quét bảng khoảng 24 lần.
- **OP4**: điều kiện hội ba vế. Bộ tối ưu so sánh dùng chỉ mục `Dno` (83), chỉ mục cụm `Salary` (1.003), chỉ mục `Sex` (5.002) và quét bảng (2.000),
  chọn chỉ mục `Dno` vì vế đó **chọn lọc nhất**, hai vế còn lại kiểm tra trong bộ nhớ.

## 4. Hàm chi phí cho phép nối (mục 19.5)

Ký hiệu: R là bảng **ngoài** (outer), S là bảng **trong** (inner), `js` là độ chọn lọc của phép nối, `bfrRS` là hệ số khối của tệp kết quả.
Mọi công thức đều cộng chi phí **ghi kết quả** ra đĩa: `⌈js × r_R × r_S / bfrRS⌉`.

| Ký hiệu | Giải thuật | Chi phí |
|---|---|---|
| J1 | Nested-loop (3 khối bộ đệm) | `b_R + b_R × b_S + kết quả` |
| J2 | Nested-loop dùng chỉ mục trên S | `b_R + r_R × (chi phí tra chỉ mục S) + kết quả`; tra chỉ mục: chính/khoá `x + 1`, phụ không khoá `x + 1 + s`, cụm `x + ⌈s/bfr⌉` |
| J3 | Sort-merge | `b_R + b_S + chi phí sắp xếp + kết quả`, sắp xếp ngoài tốn `2b × (1 + ⌈log_(nB−1)(b/nB)⌉)` nếu tệp chưa sắp |
| J4 | Hash join | `3 × (b_R + b_S) + kết quả` |

`js = 1 / max(NDV(R.A), NDV(S.B))`; khi B là khoá của S thì `js = 1 / r_S`.

In [4]:
BFR_RS = 4     # số dòng kết quả nối trên một khối (như ví dụ của sách)
NB = 3         # số khối bộ đệm


def sort_cost(rel_name):
    t = CATALOG[rel_name]
    runs = math.ceil(t["b"] / NB)
    return 2 * t["b"] * (1 + math.ceil(math.log(runs, NB - 1))) if runs > 1 else 2 * t["b"]


def probe_cost(rel, col):
    # Chi phí tra chỉ mục trên rel.col cho MỘT giá trị khoá nối
    idx = CATALOG[rel]["index"].get(col)
    if not idx:
        return None
    t = CATALOG[rel]
    s = t["r"] / t["ndv"][col]
    if idx["kind"] == "primary" or idx.get("key"):
        return idx["x"] + 1
    if idx["kind"] == "clustering":
        return idx["x"] + math.ceil(s / t["bfr"])
    return idx["x"] + 1 + s


def join_costs(R, a, S, b):
    tR, tS = CATALOG[R], CATALOG[S]
    js = 1 / max(tR["ndv"][a], tS["ndv"][b])
    res = math.ceil(js * tR["r"] * tS["r"] / BFR_RS)
    costs = {"J1 Nested-loop": tR["b"] + tR["b"] * tS["b"] + res}
    p = probe_cost(S, b)
    if p is not None:
        costs[f"J2 Index nested-loop (chỉ mục {S}.{b})"] = tR["b"] + tR["r"] * p + res
    sorted_R = tR.get("ordered_on") == a
    sorted_S = tS.get("ordered_on") == b
    costs["J3 Sort-merge"] = (tR["b"] + tS["b"] + res
                              + (0 if sorted_R else sort_cost(R)) + (0 if sorted_S else sort_cost(S)))
    costs["J4 Hash join"] = 3 * (tR["b"] + tS["b"]) + res
    return js, res, costs


def best_join(title, A, a, B, b):
    print(title)
    options = []
    for (R, ra, S, sb) in [(A, a, B, b), (B, b, A, a)]:
        js, res, costs = join_costs(R, ra, S, sb)
        options += [(f"ngoài = {R:<10}", name, c) for name, c in costs.items()]
    print(f"   js = {js:.6g}, kết quả {js * CATALOG[A]['r'] * CATALOG[B]['r']:,.0f} dòng = {res:,} khối")
    best = min(options, key=lambda o: o[2])
    for outer, name, c in sorted(options, key=lambda o: o[2]):
        print(f"   {outer} {name:<52}{c:>10,.0f} khối{'  <== chọn' if (outer, name, c) == best else ''}")
    print()


best_join("OP6: EMPLOYEE ⋈ Dno = Dnumber DEPARTMENT", "EMPLOYEE", "Dno", "DEPARTMENT", "Dnumber")
best_join("OP7: DEPARTMENT ⋈ Mgr_ssn = Ssn EMPLOYEE", "DEPARTMENT", "Mgr_ssn", "EMPLOYEE", "Ssn")

OP6: EMPLOYEE ⋈ Dno = Dnumber DEPARTMENT
   js = 0.008, kết quả 10,000 dòng = 2,500 khối
   ngoài = EMPLOYEE   J4 Hash join                                             8,539 khối  <== chọn
   ngoài = DEPARTMENT J4 Hash join                                             8,539 khối
   ngoài = DEPARTMENT J2 Index nested-loop (chỉ mục EMPLOYEE.Dno)             12,888 khối
   ngoài = EMPLOYEE   J2 Index nested-loop (chỉ mục DEPARTMENT.Dnumber)       24,500 khối
   ngoài = DEPARTMENT J1 Nested-loop                                          28,513 khối
   ngoài = EMPLOYEE   J1 Nested-loop                                          30,500 khối
   ngoài = EMPLOYEE   J3 Sort-merge                                           48,513 khối
   ngoài = DEPARTMENT J3 Sort-merge                                           48,513 khối

OP7: DEPARTMENT ⋈ Mgr_ssn = Ssn EMPLOYEE
   js = 0.0001, kết quả 125 dòng = 32 khối
   ngoài = DEPARTMENT J2 Index nested-loop (chỉ mục EMPLOYEE.Ssn)                670 khối  <== c

**Nhận xét:**

- **OP6** (mỗi nhân viên nối với phòng của mình, kết quả 10.000 dòng): hash join rẻ nhất. Nested-loop đơn thuần đắt gấp hơn 3 lần.
  Chọn bảng nào làm bảng ngoài cũng quan trọng: với J1, đặt bảng **nhỏ** (DEPARTMENT) làm bảng ngoài rẻ hơn.
- **OP7** (tìm trưởng phòng, kết quả chỉ 125 dòng): kết quả đảo ngược. Chỉ cần đọc 125 phòng ban rồi tra chỉ mục `Ssn` 125 lần,
  nên **J2 với DEPARTMENT làm bảng ngoài** rẻ nhất, rẻ hơn hash join gần 10 lần.
- Cùng hai bảng, cùng các giải thuật, nhưng **giải thuật tốt nhất phụ thuộc vào độ chọn lọc của phép nối và chỉ mục có sẵn**.
  Đó là lý do cần ước lượng chi phí thay vì dùng một quy tắc cố định.

## 5. Chọn thứ tự nối bằng quy hoạch động (mục 19.5 – 19.6)

**Câu truy vấn** (4 bảng): họ nhân viên phòng *Research* và tên dự án đặt tại *Stafford* mà họ tham gia.

```sql
SELECT e.lname, p.pname
FROM   employee e, works_on w, project p, department d
WHERE  e.ssn = w.essn AND w.pno = p.pnumber AND e.dno = d.dnumber
  AND  d.dname = 'Research' AND p.plocation = 'Stafford';
```

Với n bảng có n! thứ tự nối cho cây lệch trái (left-deep). Bộ tối ưu dùng **quy hoạch động** như System R:

1. Mức 1: với từng bảng, chọn đường truy xuất rẻ nhất cho các điều kiện chọn của nó (dùng `best_select` ở phần 3).
2. Mức k: với mỗi tập k bảng, kế hoạch tốt nhất = min trên mọi cách lấy **kế hoạch tốt nhất của k−1 bảng** rồi nối thêm 1 bảng, với giải thuật nối rẻ nhất.
3. Chỉ xét các bảng có điều kiện nối với tập trước đó: **không tạo tích Descartes** (một quy tắc kinh nghiệm được giữ lại).

Mô hình chi phí theo sách: kết quả trung gian được **ghi ra tệp tạm** rồi đọc lại ở bước sau. Khi nối kết quả trung gian L (bảng ngoài) với bảng gốc R:

| Giải thuật | Chi phí |
|---|---|
| J1 | `b_L + b_L × b_R + kết quả` (mỗi khối của L quét lại toàn bộ R, lọc điều kiện của R ngay khi đọc) |
| J2 | `b_L + r_L × (chi phí tra chỉ mục R) + kết quả` |
| J4 | `b_L + (chi phí chọn trên R) + 2 × (b_L + b_R') + kết quả` (đọc hai đầu vào, ghi rồi đọc lại các phân hoạch; R' là R sau khi lọc) |

In [5]:
QUERY = {
    "conds": {"EMPLOYEE": [], "WORKS_ON": [], "DEPARTMENT": [("Dname", "=", "Research")],
              "PROJECT": [("Plocation", "=", "Stafford")]},
    "joins": [("EMPLOYEE", "Ssn", "WORKS_ON", "Essn"),
              ("WORKS_ON", "Pno", "PROJECT", "Pnumber"),
              ("EMPLOYEE", "Dno", "DEPARTMENT", "Dnumber")],
}
ABBR = {"EMPLOYEE": "E", "WORKS_ON": "W", "PROJECT": "P", "DEPARTMENT": "D"}


def base_plan(R):
    t = CATALOG[R]
    conds = QUERY["conds"][R]
    name, on, cost = best_select(R, conds, verbose=False) if conds else ("S1 Quét tuyến tính", "", t["b"])
    rows = selectivity_and(R, conds) * t["r"]
    blocks = math.ceil(rows / t["bfr"])
    return {"tree": ABBR[R], "rows": rows, "blocks": blocks, "cost": cost + blocks,
            "steps": [f"{ABBR[R]}: {name} {on} ({cost:,.0f}) + ghi tạm {blocks:,} khối"]}


def join_predicate(left_rels, R):
    for (A, a, B, b) in QUERY["joins"]:
        if B == R and A in left_rels:
            return A, a, b
        if A == R and B in left_rels:
            return B, b, a
    return None


def join_plan(L, left_rels, R):
    # Nối kế hoạch L (bảng ngoài) với bảng gốc R (bảng trong), chọn giải thuật rẻ nhất
    tR = CATALOG[R]
    pred = join_predicate(left_rels, R)
    conds = QUERY["conds"][R]
    rR = selectivity_and(R, conds) * tR["r"]                  # số dòng R sau khi lọc
    bR = math.ceil(rR / tR["bfr"])
    if pred is None:                                           # tích Descartes
        js, label = 1.0, "×"
    else:
        A, a, b = pred
        js, label = 1 / max(CATALOG[A]["ndv"][a], tR["ndv"][b]), f"⋈ {ABBR[A]}.{a}={ABBR[R]}.{b}"
    rows = L["rows"] * rR * js
    res = math.ceil(rows / BFR_RS)
    options = {"J1": L["blocks"] + L["blocks"] * tR["b"] + res}
    if pred is not None:
        p = probe_cost(R, b)
        if p is not None:
            options["J2"] = L["blocks"] + L["rows"] * p + res
        sel_cost = best_select(R, conds, verbose=False)[2] if conds else tR["b"]
        options["J4"] = L["blocks"] + sel_cost + 2 * (L["blocks"] + bR) + res
    algo = min(options, key=options.get)
    return {"tree": f"({L['tree']} {label} {ABBR[R]})", "rows": rows, "blocks": res,
            "cost": L["cost"] + options[algo],
            "steps": L["steps"] + [f"{label} {ABBR[R]}: {algo} ({options[algo]:,.0f}), "
                                   f"kết quả ~{rows:,.0f} dòng = {res:,} khối"]}


def dynamic_programming(rels, verbose=True):
    best = {frozenset([R]): base_plan(R) for R in rels}
    if verbose:
        print("MỨC 1: truy xuất từng bảng")
        for S, plan in best.items():
            print(f"   {{{plan['tree']}}}: chi phí {plan['cost']:>10,.0f}   {plan['steps'][0]}")
    for k in range(2, len(rels) + 1):
        if verbose:
            print(f"\nMỨC {k}:")
        for combo in combinations(rels, k):
            S = frozenset(combo)
            candidates = [join_plan(best[S - {R}], S - {R}, R)
                          for R in S if S - {R} in best and join_predicate(S - {R}, R)]
            if not candidates:
                continue                                       # tập không liên thông: bỏ
            best[S] = min(candidates, key=lambda p: p["cost"])
            if verbose:
                tried = ", ".join(f"{c['tree']}={c['cost']:,.0f}" for c in candidates)
                print(f"   {{{','.join(ABBR[r] for r in combo)}}}: thử {tried}")
                print(f"      -> giữ {best[S]['tree']}")
    return best[frozenset(rels)]


RELS = list(QUERY["conds"])
best_plan = dynamic_programming(RELS)
print(f"\nKẾ HOẠCH TỐT NHẤT: {best_plan['tree']}   tổng chi phí {best_plan['cost']:,.0f} khối")
for s in best_plan["steps"]:
    print("   ", s)

MỨC 1: truy xuất từng bảng
   {E}: chi phí      4,000   E: S1 Quét tuyến tính  (2,000) + ghi tạm 2,000 khối
   {W}: chi phí      2,000   W: S1 Quét tuyến tính  (1,000) + ghi tạm 1,000 khối
   {D}: chi phí          8   D: S1 Quét tuyến tính  (6) + ghi tạm 1 khối
   {P}: chi phí         14   P: S6 Chỉ mục phụ trên Plocation (13) + ghi tạm 1 khối

MỨC 2:
   {E,W}: thử (E ⋈ E.Ssn=W.Essn W)=23,000, (W ⋈ W.Essn=E.Ssn E)=21,000
      -> giữ (W ⋈ W.Essn=E.Ssn E)
   {E,D}: thử (E ⋈ E.Dno=D.Dnumber D)=10,028, (D ⋈ D.Dnumber=E.Dno E)=112
      -> giữ (D ⋈ D.Dnumber=E.Dno E)
   {W,P}: thử (W ⋈ W.Pno=P.Pnumber P)=5,065, (P ⋈ P.Pnumber=W.Pno W)=1,065
      -> giữ (P ⋈ P.Pnumber=W.Pno W)

MỨC 3:
   {E,W,D}: thử ((W ⋈ W.Essn=E.Ssn E) ⋈ E.Dno=D.Dnumber D)=51,088, ((D ⋈ D.Dnumber=E.Dno E) ⋈ E.Ssn=W.Essn W)=532
      -> giữ ((D ⋈ D.Dnumber=E.Dno E) ⋈ E.Ssn=W.Essn W)
   {E,W,P}: thử ((W ⋈ W.Essn=E.Ssn E) ⋈ W.Pno=P.Pnumber P)=51,065, ((P ⋈ P.Pnumber=W.Pno W) ⋈ W.Essn=E.Ssn E)=2,165
      -> giữ ((P ⋈ P.Pnu

### Kiểm chứng: liệt kê đủ 4! = 24 thứ tự nối

Quy hoạch động chỉ giữ **một kế hoạch tốt nhất cho mỗi tập bảng** nên không phải thử hết mọi thứ tự.
Ô dưới đây vét cạn cả 24 thứ tự (kể cả thứ tự phải dùng tích Descartes) để kiểm tra quy hoạch động đã chọn đúng.

In [6]:
def plan_for_order(order):
    plan, left = base_plan(order[0]), {order[0]}
    for R in order[1:]:
        plan = join_plan(plan, left, R)
        left.add(R)
    return plan


all_plans = sorted((plan_for_order(o) for o in permutations(RELS)), key=lambda p: p["cost"])
print(f"{'Hạng':<6}{'Thứ tự nối (cây lệch trái)':<68}{'Chi phí (khối)':>16}")
for i, p in enumerate(all_plans, 1):
    if i <= 6 or i > len(all_plans) - 3:
        print(f"{i:<6}{p['tree']:<68}{p['cost']:>16,.0f}")
    elif i == 7:
        print("...")

worst = all_plans[-1]
print(f"\nQuy hoạch động chọn đúng kế hoạch rẻ nhất: {best_plan['cost'] == all_plans[0]['cost']}")
print(f"Kế hoạch tệ nhất đắt gấp {worst['cost'] / best_plan['cost']:,.0f} lần kế hoạch tốt nhất")
cartesian = sum("×" in p["tree"] for p in all_plans)
print(f"{cartesian}/{len(all_plans)} thứ tự phải dùng tích Descartes; quy hoạch động bỏ qua chúng ngay từ đầu")

Hạng  Thứ tự nối (cây lệch trái)                                            Chi phí (khối)
1     (((D ⋈ D.Dnumber=E.Dno E) ⋈ E.Ssn=W.Essn W) ⋈ W.Pno=P.Pnumber P)                 788
2     (((P ⋈ P.Pnumber=W.Pno W) ⋈ W.Essn=E.Ssn E) ⋈ E.Dno=D.Dnumber D)               2,324
3     (((P ⋈ P.Pnumber=W.Pno W) × D) ⋈ W.Essn=E.Ssn E)                               2,915
4     (((P × D) ⋈ P.Pnumber=W.Pno W) ⋈ W.Essn=E.Ssn E)                               4,184
5     (((D × P) ⋈ P.Pnumber=W.Pno W) ⋈ W.Essn=E.Ssn E)                               4,364
6     (((P × D) ⋈ D.Dnumber=E.Dno E) ⋈ E.Ssn=W.Essn W)                               5,264
...
22    (((P × E) ⋈ E.Ssn=W.Essn W) ⋈ E.Dno=D.Dnumber D)                             505,824
23    (((E × P) ⋈ E.Dno=D.Dnumber D) ⋈ E.Ssn=W.Essn W)                             510,408
24    (((E × P) ⋈ E.Ssn=W.Essn W) ⋈ E.Dno=D.Dnumber D)                             909,808

Quy hoạch động chọn đúng kế hoạch rẻ nhất: True
Kế hoạch tệ nhất đắt gấp 1,155 lần kế

**Nhận xét:**

- Kế hoạch tốt nhất bắt đầu từ DEPARTMENT, bảng **chọn lọc mạnh nhất** (chỉ còn 1 dòng sau khi lọc), nên kết quả trung gian nhỏ ngay từ đầu
  (80 → 320 → 2 dòng). Nó tra **chỉ mục** (J2) vào EMPLOYEE và WORKS_ON thay vì quét hai bảng lớn. PROJECT sau khi lọc chỉ còn 10 dòng nên được nối cuối bằng hash join.
- Cùng một câu SQL, cùng kết quả, nhưng chọn sai thứ tự nối có thể đắt hơn hơn 1.000 lần. Quy tắc kinh nghiệm
  "đẩy phép chọn xuống" (notebook 1) **không đủ** để biết nên nối bảng nào trước và dùng giải thuật nào: cần ước lượng chi phí.
- Quy hoạch động cho cùng kết quả với vét cạn nhưng chỉ xét các tập con liên thông; với n lớn, n! thứ tự là không thể thử hết.
- Để ý các thứ tự hạng 3 – 6: dù dùng tích Descartes, chúng vẫn khá rẻ vì `P × D` sau khi lọc chỉ có 10 × 1 dòng.
  Quy tắc "không tạo tích Descartes" là một **quy tắc kinh nghiệm**, có thể bỏ sót kế hoạch tốt. Lược đồ hình sao (notebook 6) khai thác đúng trường hợp này:
  nhân các bảng chiều nhỏ đã lọc với nhau trước rồi mới tra vào bảng sự kiện.
- Toàn bộ quyết định dựa trên **thống kê ước lượng**. Nếu thống kê sai (ví dụ dữ liệu lệch, hoặc thống kê đã cũ), kế hoạch "rẻ nhất trên giấy" có thể chạy rất chậm.
  Phần đối chiếu PostgreSQL bên dưới cho thấy điều này.

## 6. Đối chiếu trên PostgreSQL

Mở file `2.2_cbo_postgres_demo.sql` trong pgAdmin (CSDL `company`, đã chạy file 01 và 02) và chạy từng khối.

| Ý trong notebook | Demo trong file SQL | Cần chỉ ra |
|---|---|---|
| Catalog `r`, `b`, `NDV` | Demo 1 | `pg_class.reltuples`, `relpages`; `pg_stats.n_distinct`, `most_common_vals` |
| Hàm chi phí | Demo 2 | Tự tính `cost` của Seq Scan từ `relpages`, `reltuples` khớp đúng với `EXPLAIN` |
| Độ chọn lọc | Demo 3 | `rows` ước lượng của `dno = 5` và `dno = 17` lấy từ tần suất trong `most_common_freqs` |
| Có chỉ mục chưa chắc dùng (OP2 / OP3) | Demo 4 | cùng câu, đổi hằng số `dno = 5` → Seq Scan, `dno = 17` → Bitmap Index Scan |
| Thống kê sai → kế hoạch sai | Demo 5 | `rows` ước lượng lệch hẳn thực tế, `ANALYZE` xong thì kế hoạch đổi |
| Thứ tự nối | Demo 6 | ép sai thứ tự bằng `join_collapse_limit = 1`, so sánh `cost` và `Execution Time` |

> *(Dán ảnh kết quả EXPLAIN vào đây khi viết báo cáo)*

## 7. Kết luận

1. Bộ tối ưu dựa trên chi phí dùng **thống kê trong catalog** để ước lượng độ chọn lọc, số dòng trung gian và chi phí của mỗi kế hoạch.
2. Với phép chọn, chỉ mục chỉ có lợi khi điều kiện đủ chọn lọc (OP3); với điều kiện lấy lên nhiều dòng, quét tuần tự rẻ hơn (OP2).
3. Với phép nối, giải thuật và bảng ngoài tốt nhất phụ thuộc vào kích thước bảng, độ chọn lọc của phép nối và chỉ mục có sẵn (OP6 và OP7).
4. Thứ tự nối được chọn bằng quy hoạch động, giúp tránh kế hoạch tệ đắt hơn hàng nghìn lần mà không phải thử n! thứ tự.
5. Chất lượng kế hoạch phụ thuộc vào **độ chính xác của thống kê**, nên các hệ quản trị như PostgreSQL cần chạy `ANALYZE` thường xuyên.